## Rule-based data integration pipeline (standard blocking)

In [1]:
from pathlib import Path

ROOT = Path.cwd()

DATA_DIR = ROOT / "parquet"
OUTPUT_DIR = ROOT / "output"
MLDS_DIR = ROOT / "ml-datasets"
BLOCK_EVAL_DIR = OUTPUT_DIR / "blocking_evaluation"
CORR_DIR = OUTPUT_DIR / "correspondences"

PIPELINE_DIR = CORR_DIR / "pipeline_corr"
PIPELINE_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
from PyDI.io import load_parquet

amazon_books = load_parquet(
    DATA_DIR / "amazon_new.parquet",
    name="amazon_books"
)

goodreads = load_parquet(
    DATA_DIR / "goodreads_new.parquet",
    name="goodreads"
)

metabooks = load_parquet(
  DATA_DIR / "metabooks_new.parquet",
  name="metabooks"
)

In [3]:
from PyDI.entitymatching import StandardBlocker
from PyDI.entitymatching import StringComparator, NumericComparator
from PyDI.entitymatching import RuleBasedMatcher


# standard blocker: metabooks > amazon
st_blocker_m2a = StandardBlocker(
    metabooks, amazon_books,
    on=['title_norm','author_norm'],
    batch_size=1000,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id'
)

# standard blocker: metabooks > goodreads
st_blocker_m2g = StandardBlocker(
    metabooks, goodreads,
    on=['title_norm','author_norm'],
    batch_size=1000,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id'
)

/Users/abd/Developer/team_project_data_integration/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
comparators_m2a = [
    StringComparator(column="title_norm", similarity_function="cosine"),
    StringComparator(column="author_norm", similarity_function="cosine", preprocess=str.lower),
    NumericComparator(column="publish_year", max_difference=1),
]

comparators_m2g = comparators_m2a + [
    StringComparator(column="genres", similarity_function="jaccard",
                     preprocess=str.lower, list_strategy="concatenate"),
    NumericComparator(column="price", max_difference=5),
    NumericComparator(column="page_count", max_difference=10),
    NumericComparator(column="rating", max_difference=0.2),
]

In [5]:
matcher = RuleBasedMatcher()

# matching metabooks > amazon
st_corr_m2a = matcher.match(
    df_left=metabooks,
    df_right=amazon_books, 
    candidates=st_blocker_m2a,
    comparators=comparators_m2a,
    weights=[0.4, 0.4, 0.2], 
    threshold=0.7,
    id_column='id'
)

# matching metabooks > goodreads
st_corr_m2g = matcher.match(
    df_left=metabooks,
    df_right=goodreads, 
    candidates=st_blocker_m2g,
    comparators=comparators_m2g,
    weights=[0.5, 0.25, 0.05, 0.05, 0.05, 0.05, 0.05], 
    threshold=0.7,
    id_column='id'
)

In [7]:
st_corr_m2a.to_csv(PIPELINE_DIR / "st_corr_m2a.csv", index=False)
st_corr_m2g.to_csv(PIPELINE_DIR / "st_corr_m2g.csv", index=False)

In [8]:
# data fusion for standard blocker
from PyDI.fusion import DataFusionStrategy, DataFusionEngine, longest_string, union, prefer_higher_trust
import pandas as pd

amazon_books.attrs["trust_score"] = 3
metabooks.attrs["trust_score"] = 2
goodreads.attrs["trust_score"] = 1

all_standard_correspondences = pd.concat([st_corr_m2a, st_corr_m2g], ignore_index=True)

len(all_standard_correspondences)

60151

In [9]:
strategy = DataFusionStrategy('books_fusion_strategy')

strategy.add_attribute_fuser('title_norm', longest_string)
strategy.add_attribute_fuser('author_norm', longest_string)
strategy.add_attribute_fuser('publish_year', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('price', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('page_count', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('rating', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('genres', union)

In [10]:
engine = DataFusionEngine(strategy, debug=True, debug_format='json',
                          debug_file= PIPELINE_DIR / "debug_fusion_standard_blocker.jsonl")

fused_standard_blocker = engine.run(
    datasets=[amazon_books, metabooks, goodreads],
    correspondences=all_standard_correspondences,
    id_column="id",
    include_singletons=False,
)

print(f'Fused rows: {len(fused_standard_blocker):,}')
display(fused_standard_blocker.head())

Fused rows: 25,214


,_id,_fusion_group_id,_fusion_sources,publish_year,edition,publisher,title_norm,rating,rating_number,price,...,isbn_clean,title,id,author_norm,page_count,language,author,genres,_fusion_confidence,_fusion_metadata
0,goodreads_46097,group_0,"[metabooks, goodreads]",2006.0,nan,Ballantine Books,in the dark of the night,4.5,4775,4.990000,...,034548701X,In the Dark of the Night,goodreads_46097,john saul,324.0,English,John Saul,"[['Horror', 'Fiction', 'Mystery', 'Thriller', ...",0.666667,"{'publish_year_rule': 'prefer_higher_trust', '..."
1,goodreads_3677,group_1,"[metabooks, goodreads]",2006.0,nan,HarperCollins,midnight,4.8,33466,15.940000,...,0060744510,Midnight,goodreads_3677,erin hunter,368.0,English,Erin Hunter,"[[""Children's Books"", 'Literature', 'Fiction']...",0.666667,"{'publish_year_rule': 'prefer_higher_trust', '..."
2,metabooks_12970,group_2,"[metabooks, amazon_books]",1990.0,NaN,E. P. Dutton,the killing man,4.3,106,13.800000,...,0318668556,The Killing Man,metabooks_12970,mickey spillane,228.0,English,Mickey Spillane,"[['Mystery, Thriller', 'Suspense', 'Mystery']]",0.653846,"{'publish_year_rule': 'prefer_higher_trust', '..."
3,amazon_211764,group_3,"[metabooks, amazon_books]",1983.0,NaN,Ace Books,the deadly streets,4.3,31,6.500000,...,0441142184,The Deadly Streets,amazon_211764,harlan ellison,208.0,English,Harlan Ellison,"[['Literature', 'Fiction']]",0.692308,"{'publish_year_rule': 'prefer_higher_trust', '..."
4,amazon_170033,group_4,"[metabooks, amazon_books]",1986.0,NaN,Harpercollins,i tina,4.5,271,57.389999,...,068805949X,"I, Tina",amazon_170033,tina turner,288.0,English,Tina Turner,"[['Arts', 'Photography', 'Music']]",0.653846,"{'publish_year_rule': 'prefer_higher_trust', '..."


In [11]:
fused_standard_blocker.to_csv(PIPELINE_DIR / "fused_standard_blocker.csv")